# 🎤 NVIDIA Canary-Qwen-2.5B 語音轉文字轉錄器
**SOTA 英語 ASR** — WER 5.63% | FastConformer + Qwen3-1.7B

⚠️ **English-only** · T4 (免費) 可跑 · 第一步安裝約 8-15 分鐘 · 第一步完成後會自動重啟 Runtime，重啟後直接跑第二步

In [ ]:
# @title 🛠️ 第一步：安裝環境 (完成後自動重啟 Runtime)
import subprocess, sys, os, time

def run(cmd, desc):
    print(f"  ⏳ {desc}...", end="", flush=True)
    t0 = time.time()
    r = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if r.returncode != 0:
        print(f" ❌ ({time.time()-t0:.0f}s)")
        for line in r.stdout.strip().split('\n')[-15:]:
            print(f"     {line}")
        raise RuntimeError(desc)
    print(f" ✅ ({time.time()-t0:.0f}s)")

P = [sys.executable, "-m", "pip"]
print("📦 安裝中...\n")
run(["apt-get","-qq","-y","update"], "apt update")
run(["apt-get","-qq","-y","install","ffmpeg","libsndfile1"], "ffmpeg")
run(P+["install","-q","Cython","packaging","pydub","soundfile","huggingface_hub"], "Python 依賴")
run(P+["install","-q","nemo_toolkit[asr,tts] @ git+https://github.com/NVIDIA-NeMo/NeMo.git"], "NeMo (git main)")
run(P+["uninstall","-y","numpy","scipy","numba","librosa","llvmlite"], "移除舊 NumPy")
run(P+["install","-q","--force-reinstall","--no-cache-dir","numpy","scipy","numba","librosa"], "重建 NumPy ABI")

# 驗證
r = subprocess.run([sys.executable, "-c",
    "from nemo.collections.speechlm2.models import SALM; print('  ✅ speechlm2.SALM 可用')"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
print(r.stdout.strip() if r.returncode == 0 else f"  ❌ 驗證失敗: {r.stderr.strip().split(chr(10))[-1]}")

print("\n✅ 安裝完成 — 3 秒後自動重啟 Runtime")
print("   重啟後直接跑【第二步】")
time.sleep(3)
os.kill(os.getpid(), 9)

In [ ]:
# @title 🚀 第二步：上傳音檔 → 轉錄
import os, sys, subprocess, warnings, time, gc, threading, logging
import re, inspect, tempfile
from typing import Tuple, List
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass

# === 全域靜音 ===
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["NEMO_VERBOSITY"] = "error"
warnings.filterwarnings("ignore")
for name in ["nemo_logger","transformers","huggingface_hub",
             "huggingface_hub.utils._http","filelock","fsspec"]:
    logging.getLogger(name).setLevel(logging.ERROR)

import torch
import numpy as np
import soundfile as sf
from pydub import AudioSegment

# === Qwen3 Patch (v3.2) ===
import transformers, torch.nn as _nn
_DUMMY = _nn.Embedding(1, 1)
for _cp in ['transformers.models.qwen3.modeling_qwen3.Qwen3Model',
            'transformers.models.qwen3.modeling_qwen3.Qwen3ForCausalLM']:
    try:
        _mp, _cn = _cp.rsplit('.', 1)
        _cls = getattr(__import__(_mp, fromlist=[_cn]), _cn)
        if not hasattr(_cls, '_p'):
            def _mg(d=_DUMMY):
                def g(self):
                    if hasattr(self,'embed_tokens'): return self.embed_tokens
                    m=getattr(self,'model',None)
                    if m and hasattr(m,'embed_tokens'): return m.embed_tokens
                    return d
                return g
            def _ms():
                def s(self,v):
                    if hasattr(self,'embed_tokens'): self.embed_tokens=v
                    elif hasattr(self,'model') and hasattr(self.model,'embed_tokens'): self.model.embed_tokens=v
                return s
            _cls.get_input_embeddings=_mg(); _cls.set_input_embeddings=_ms(); _cls._p=True
    except: pass
if not hasattr(transformers.PreTrainedModel, '_p'):
    _og = transformers.PreTrainedModel.get_input_embeddings
    def _sg(self, _d=_DUMMY):
        try: return _og(self)
        except (NotImplementedError, AttributeError):
            if hasattr(self,'embed_tokens'): return self.embed_tokens
            m=getattr(self,'model',None)
            if m and hasattr(m,'embed_tokens'): return m.embed_tokens
            return _d
    transformers.PreTrainedModel.get_input_embeddings=_sg; transformers.PreTrainedModel._p=True

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# === 資料結構 ===
@dataclass
class Seg:
    s: float; e: float; text: str
    def _fmt(self, t, srt=False):
        t=max(0.0,t); h,r=divmod(int(t),3600); m,s=divmod(r,60)
        return f"{h:02d}:{m:02d}:{s:02d},{int((t-int(t))*1000):03d}" if srt else f"{h:02d}:{m:02d}:{s:02d}"
    def line(self): return f"[{self._fmt(self.s)} → {self._fmt(self.e)}] {self.text}"
    def srt(self, i): return f"{i}\n{self._fmt(self.s,True)} --> {self._fmt(self.e,True)}\n{self.text}\n"

def dedup(prev, curr, n=10):
    if not prev or not curr: return curr
    wp,wc = prev.split(), curr.split()
    f=lambda w: re.sub(r'[^\w\s]','',w).lower()
    a,b = [f(w) for w in wp], [f(w) for w in wc]
    k=0
    for i in range(1, min(len(a),len(b),n)+1):
        if a[-i:]==b[:i]: k=i
    return ' '.join(wc[k:]) if k else curr

# === 音訊處理 ===
def convert_audio(src):
    fd,wav=tempfile.mkstemp(suffix='.wav'); os.close(fd)
    try:
        subprocess.run(['ffmpeg','-y','-threads','2','-i',src,'-ar','16000','-ac','1','-c:a','pcm_s16le','-loglevel','error',wav],check=True,capture_output=True)
    except:
        AudioSegment.from_file(src).set_frame_rate(16000).set_channels(1).set_sample_width(2).export(wav,format='wav')
    return wav, sf.info(wav).duration

def split_audio(wav, dur, maxs=35.0, overlap=1.0):
    if dur <= maxs+5: return [(wav,0.0,dur)]
    data,sr = sf.read(wav)
    chunks,s = [],0.0
    while s<dur:
        e=min(s+maxs,dur); fd,cp=tempfile.mkstemp(suffix='.wav'); os.close(fd)
        sf.write(cp,data[int(s*sr):int(e*sr)],sr); chunks.append((cp,s,e)); s+=maxs-overlap
    return chunks

# === ASR ===
class ASR:
    def __init__(self, dev, dt, bs):
        self.dev,self.dt,self.bs = dev,dt,bs
        self.model=None; self._ok=False; self._st='初始化'
        self._mnt=False; self._tag='<|audioplaceholder|>'

    def load(self):
        if self._ok: return self
        self._st='下載模型中...'
        from nemo.collections.speechlm2.models import SALM
        self.model = SALM.from_pretrained('nvidia/canary-qwen-2.5b')
        if self.dev!='cpu': self.model=self.model.to(device=self.dev,dtype=self.dt)
        self.model.eval()
        self._mnt = 'max_new_tokens' in list(inspect.signature(self.model.generate).parameters)
        if hasattr(self.model,'audio_locator_tag'): self._tag=self.model.audio_locator_tag
        self._ok=True; self._st='就緒'
        return self

    @property
    def ready(self): return self._ok
    @property
    def status(self): return self._st

    @torch.inference_mode()
    def _batch(self, paths):
        ps=[[{'role':'user','content':f'Transcribe the following: {self._tag}','audio':[p]}] for p in paths]
        try:
            ids = self.model.generate(prompts=ps, max_new_tokens=512) if self._mnt else self.model.generate(prompts=ps)
        except (AssertionError, ValueError, TypeError):
            ps2=[[{'role':'user','slots':{'text':f'Transcribe the following: {self._tag}'},'audio':[p]}] for p in paths]
            ids = self.model.generate(prompts=ps2)
        return [self.model.tokenizer.ids_to_text(a.cpu()).strip() for a in ids]

    @torch.inference_mode()
    def run(self, wav, dur):
        t0=time.time(); chunks=split_audio(wav,dur); n=len(chunks)
        alive=True
        def tick():
            while alive:
                e=int(time.time()-t0); print(f'\r   ⏳ {e//60:02d}:{e%60:02d}',end='',flush=True); time.sleep(1)
        th=threading.Thread(target=tick,daemon=True); th.start()
        segs,tmps=[],set()
        try:
            for i in range(0,n,self.bs):
                b=chunks[i:i+self.bs]; bp=[c[0] for c in b]
                tmps.update(p for p in bp if p!=wav)
                try: tx=self._batch(bp)
                except:
                    tx=[]
                    for p in bp:
                        try: tx.append(self._batch([p])[0])
                        except: tx.append('')
                for j,t in enumerate(tx):
                    t=t.strip()
                    if t:
                        if segs: t=dedup(segs[-1].text,t)
                        if t: segs.append(Seg(b[j][1],b[j][2],t))
                for p in bp:
                    if p!=wav:
                        try: os.remove(p); tmps.discard(p)
                        except: pass
                if torch.cuda.is_available(): torch.cuda.empty_cache()
        finally:
            alive=False; th.join(timeout=2); print('\r'+' '*30+'\r',end='')
            for p in tmps:
                try: os.remove(p)
                except: pass
        el=time.time()-t0; rtf=dur/el if el>0 else 0
        return segs, el, rtf

# === Main ===
def main():
    print('🎤 Canary-Qwen-2.5B 轉錄器\n')
    wav=None; asr=None; pool=None
    try:
        # GPU
        if not torch.cuda.is_available():
            print('⚠️ 無 GPU'); dev,dt,bs='cpu',torch.float32,1
        else:
            name=torch.cuda.get_device_name(0)
            vram=torch.cuda.get_device_properties(0).total_memory/(1024**3)
            bs=16 if vram>=38 else (8 if vram>=22 else 4)
            dt=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
            dev='cuda:0'
            print(f'🖥️ {name} ({vram:.0f}GB) | batch={bs}')

        asr=ASR(dev,dt,bs)
        pool=ThreadPoolExecutor(max_workers=2)
        fut=pool.submit(asr.load)

        print('📂 請上傳英語音檔：\n')
        if IN_COLAB:
            uploaded=files.upload()
            if not uploaded: return
            src=list(uploaded.keys())[0]
        else:
            src=input('路徑: ')
            if not os.path.exists(src): return

        wav,dur=convert_audio(src)
        print(f'📎 {src} ({dur:.1f}s)')

        if not asr.ready:
            print('⏳ 載入模型...',end='',flush=True)
            while not asr.ready: time.sleep(1)
            print(' ✅')
        fut.result()

        segs,el,rtf = asr.run(wav,dur)
        print(f'✅ {el:.1f}s ({rtf:.1f}x RTFx)\n')

        if not segs:
            print('⚠️ 無轉錄結果'); return

        # 輸出
        bd='/content' if IN_COLAB else os.getcwd()
        bn=os.path.splitext(os.path.basename(src))[0]
        tp=os.path.join(bd,f'{bn}_transcript.txt')
        sp=os.path.join(bd,f'{bn}_subtitle.srt')
        with open(tp,'w',encoding='utf-8') as f:
            for s in segs: f.write(s.line()+'\n')
        with open(sp,'w',encoding='utf-8') as f:
            for i,s in enumerate(segs,1): f.write(s.srt(i)+'\n')

        # 預覽
        for s in segs[:10]: print(s.line())
        if len(segs)>10: print(f'... 共 {len(segs)} 段')
        print(f'\n💾 {tp}\n💾 {sp}')

        if IN_COLAB:
            files.download(tp); files.download(sp)

    except Exception as e:
        print(f'\n❌ {type(e).__name__}: {e}')
        import traceback; traceback.print_exc()
    finally:
        if wav and os.path.exists(wav):
            try: os.remove(wav)
            except: pass
        if pool: pool.shutdown(wait=False)
        if asr and asr.model: del asr.model
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        gc.collect()

main()